# Ćw. 11a Modele transformerów. Parametry generowania tekstu.

Modele transformerów są obecnie standardowymi modelami używanymi w zadaniach NLP. Prawie wszystkie zadania NLP obejmują generowanie tekstu, ale nie jest on bezpośrednim wynikiem działania modelu. Można oczekiwać, że model pomoże wygenerować tekst, który jest spójny i kontekstowo odpowiedni. Chociaż częściowo jest to związane z jakością modelu, parametry generowania również odgrywają kluczową rolę w jakości generowanego tekstu.

Na tych ćwiczeniach poznasz kluczowe parametry, które kontrolują generowanie tekstu w modelach transformatorowych. Zobaczysz, jak te parametry wpływają na jakość generowanego tekstu i jak je dostroić do różnych zastosowań. W szczególności dowiesz się:

* Jakie są podstawowe parametry kontrolujące generowanie tekstu w modelach
transformatorów
* Jakie są różne strategie dekodowania
* Jak kontrolować kreatywność i spójność generowanego tekstu
* Jak dostroić parametry generowania dla konkretnych aplikacji


# Zawartość ćwiczeń

* Podstawowe parametry generowania tekstu
* Eksperymentowanie z temperaturą
* Próbkowanie Top-K i Top-P
* Kontrolowanie powtarzalności
* Zachłanne dekodowanie i próbkowanie
* Parametry dla konkretnych zastosowań
* Wyszukiwanie wiązki i generowanie wielu sekwencji


##  Podstawowe parametry generowania tekstu


Jako przykład wybierzmy model GPT-2. Jest to mały model transformatora, który nie wymaga dużych zasobów obliczeniowych, ale nadal jest w stanie generować wysokiej jakości tekst. Prosty przykład generowania tekstu przy użyciu modelu GPT-2 jest następujący:

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Tworzenie modelu i tokenizatora
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Tokenizacja promptu/monitu wejściowego do sekwencji identyfikatorów
prompt = "Artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")
# generowanie outputu jako ciąg  token ids
output = model.generate(
    **inputs,
    max_length=50,
    num_return_sequences=1,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
# przekształcenie token ids na text strings
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(f"Prompt: {prompt}")
print("Generated Text:")
print(generated_text)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Prompt: Artificial intelligence is
Generated Text:
Artificial intelligence is becoming better at knowing when your mind is getting too overwhelmed. We can now create "brain waves" where our brains are constantly processing inputs that you can't get your head around. For example, when the brain tries to process a


Podałeś monit składający się tylko z trzech słów, a model wygenerował długi fragment tekstu. Nie jest to generowane w jednym ujęciu, ale model jest wywoływany wiele razy w procesie iteracyjnym.

Możesz zobaczyć liczne parametry używane w funkcji generate(). Pierwszym, którego użyłeś, jest max_length. Trywialnie rzecz ujmując, kontroluje to, jak długi powinien być wygenerowany tekst, w liczbie tokenów. Zazwyczaj model generuje jeden token naraz przy użyciu monitu jako kontekstu. Następnie dołącza nowo wygenerowany token do monitu i wygeneruje następny token. Dlatego im dłuższy ma być wygenerowany tekst, tym więcej czasu zajmuje jego wygenerowanie. Zauważ, że chodzi o tokeny, a nie słowa, ponieważ użyłeś tokenizatora podsłów z modelem GPT-2. Jeden token może być tylko jednostką podwyrazu, a nie pełnym słowem.

Jednak model nie generuje konkretnie żadnego pojedynczego tokena. Zamiast tego generuje "logit", który jest wektorem prawdopodobieństwa następnego tokena. Logit jest długim wektorem, dokładnie tak długim, jak rozmiar słownictwa. Biorąc pod uwagę, że jest to rozkład prawdopodobieństwa na wszystkie możliwe "następne tokeny", możesz wybrać token o najwyższym prawdopodobieństwie (gdy ustawisz do_sample=False) lub dowolny inny token o niezerowym prawdopodobieństwie (gdy ustawisz do_sample=True). Do tego służą wszystkie inne parametry.

Parametr temperatury zniekształca rozkład prawdopodobieństwa. Niższa temperatura podkreśla najbardziej prawdopodobny token, podczas gdy wyższa temperatura zmniejsza różnicę między prawdopodobnym a mało prawdopodobnym tokenem. Domyślna temperatura to 1,0 i powinna być wartością dodatnią. Parametr top_k wybiera wtedy tylko k górnych
tokenów, a nie całe słownictwo tokenów. Następnie prawdopodobieństwo jest przeliczane na sumę do 1. Następnie, jeśli top_p jest ustawiona, ten zestaw
k tokenów jest dalej filtrowany, aby zachować te najlepsze, które składają się na całkowite prawdopodobieństwo p. Ten końcowy zestaw tokenów jest następnie używany do próbkowania następnego tokena, a proces ten nazywa się próbkowaniem jądra.

Pamiętaj, że generujesz sekwencję tokenów, jeden po drugim. Są szanse, że na każdym kroku zobaczysz ten sam token wielokrotnie i możesz zobaczyć ten sam token wyprodukowany w sekwencji. Zwykle nie jest to to, czego chcesz, więc możesz chcieć zmniejszyć prawdopodobieństwo tych tokenów, gdy zobaczysz je ponownie. Do tego właśnie służy parametr repetition_penalty.

## Eksperymentowanie z temperaturą

Biorąc pod uwagę, że wiesz, co robią różne parametry, zobaczmy, jak zmienia się wyjście, gdy dostosujesz niektóre z nich.

Parametr temperatury ma znaczący wpływ na kreatywność i losowość generowanego tekstu. Możesz zobaczyć jego efekt na poniższym przykładzie:

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text with different temperature values
temperatures = [0.2, 0.5, 1.0, 1.5]
print(f"Prompt: {prompt}")
for temp in temperatures:
    print()
    print(f"Temperature: {temp}")
    output = model.generate(
        **inputs,
        max_length=100,
        num_return_sequences=1,
        temperature=temp,
        top_k=50,
        top_p=1.0,
        repetition_penalty=1.0,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print("Generated Text:")
    print(generated_text)

Prompt: The future of artificial intelligence is

Temperature: 0.2
Generated Text:
The future of artificial intelligence is in the hands of the next generation of AI.

The future of artificial intelligence is in the hands of the next generation of AI.

The future of artificial intelligence is in the hands of the next generation of AI.

The future of artificial intelligence is in the hands of the next generation of AI.

The future of artificial intelligence is in the hands of the next generation of AI.

The future of artificial intelligence is in the hands of

Temperature: 0.5
Generated Text:
The future of artificial intelligence is uncertain. Artificial intelligence will continue to evolve, but it will ultimately be limited to the most advanced algorithms and algorithms that can be used.

The future of artificial intelligence is uncertain. Artificial intelligence will continue to evolve, but it will ultimately be limited to the most advanced algorithms and algorithms that can be used. 

Przy niskiej temperaturze (np. 0,2) tekst staje się bardziej skoncentrowany i deterministyczny, często trzymając się utartych fraz i konwencjonalnych pomysłów. Widzisz również, że ciągle powtarza to samo zdanie, ponieważ prawdopodobieństwo koncentruje się na kilku tokenach, co ogranicza różnorodność. Ten problem można rozwiązać, używając parametru kary za powtórzenie, który został omówiony w poniższej sekcji.

Przy średniej temperaturze (np. od 0,5 do 1,0) tekst ma dobrą równowagę między spójnością a kreatywnością. Wygenerowany tekst może nie być oparty na faktach, ale język jest naturalny.

Przy wysokiej temperaturze (np. 1,5) tekst staje się bardziej przypadkowy i kreatywny, ale może też być mniej spójny, a czasem nielogiczny. Język może być trudny do zrozumienia, tak jak w powyższym przykładzie.

Wybór odpowiedniej temperatury zależy od zastosowania. Jeśli tworzysz pomocnika do uzupełniania lub pisania kodu, niższa temperatura jest często lepsza. W przypadku kreatywnego pisania lub burzy mózgów wyższa temperatura może powodować bardziej zróżnicowane i ciekawe wyniki.

# Top-K i Top-P Sampling

Parametry próbkowania jądra kontrolują, jak elastycznie zezwalasz modelowi na wybranie następnego tokenu. Czy należy dostosować parametr top_k czy parametr top_p? Zobaczmy ich efekt na przykładzie:

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "The best way to learn programming is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text with different top_k values
top_k_values = [5, 20, 50]
print(f"Prompt: {prompt}")

for top_k in top_k_values:
    print()
    print(f"Top-K = {top_k}")
    output = model.generate(
        **inputs,
        max_length=100,
        num_return_sequences=1,
        temperature=1.0,
        top_k=top_k,
        top_p=1.0,
        repetition_penalty=1.0,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print("Generated Text:")
    print(generated_text)

# Generate text with different top_p values
top_p_values = [0.5, 0.7, 0.9]
for top_p in top_p_values:
    print()
    print(f"Top-P = {top_p}")
    output = model.generate(
        **inputs,
        max_length=100,
        num_return_sequences=1,
        temperature=1.0,
        top_k=0,
        top_p=top_p,
        repetition_penalty=1.0,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print("Generated Text:")
    print(generated_text)

Prompt: The best way to learn programming is

Top-K = 5
Generated Text:
The best way to learn programming is to learn how to write a program, not how to write a codebase, so that you can write your own code.

If you are interested in writing a simple web application, you should start by learning programming languages. You may want to learn how to use the Internet to learn programming.

The best way to learn programming is to learn how to write a program, not how to write a codebase, so that you can write your own code

Top-K = 20
Generated Text:
The best way to learn programming is not to go and try to figure it out yourself. You're only as good as you look at things. That's where the good parts come in: in an online forum. If there's an online forum, it's usually about coding, about the world, about what happens when you go online and have a conversation about something. Or it's something you do when you are bored.

You are in there doing whatever it takes to be a programmer,

Top-K =

Widać to za pomocą małego k
np.  5, model ma mniej opcji do wyboru, co skutkuje bardziej przewidywalnym tekstem. W skrajnym przypadku, gdy k=1, model zawsze wybiera pojedynczy token z najwyższym prawdopodobieństwem, co jest zachłannym dekodowaniem i zwykle generuje słabe dane wyjściowe. Z większym k, na przykład 50, model ma więcej opcji do wyboru, co skutkuje bardziej zróżnicowanym tekstem.

Podobnie w przypadku parametru top_p mniejszy parametr
oznacza, że model wybiera z mniejszego zestawu tokenów o wysokim prawdopodobieństwie, co skutkuje bardziej skoncentrowanym tekstem. Z większym
p, na przykład 0,9, model ma szerszy wybór, co może prowadzić do bardziej zróżnicowanego tekstu. Jednak ile opcji możesz wybrać dla danego
nie jest ustalony. Zależy to od rozkładu prawdopodobieństwa zgodnie z przewidywaniami modelu. Gdy model jest bardzo pewny co do następnego tokenu (na przykład jest ograniczony przez niektóre reguły gramatyczne), dozwolony jest tylko bardzo mały zestaw tokenów. Ta adaptacyjna natura jest również powodem, dla którego próbkowanie top-p jest często preferowane zamiast próbkowania top-k.

## Kontrolowanie powtarzalności


Powtarzanie jest częstym problemem w generowaniu tekstu. Parametr repetition_penalty pomaga rozwiązać ten problem, karząc tokeny, które już pojawiły się w wygenerowanym tekście. Zobaczmy, jak to działa:


In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "Once upon a time, there was a"
inputs = tokenizer(prompt, return_tensors="pt")

# Generuj tekst z różnymi karami za powtórzenia
penalties = [1.0, 1.2, 1.5, 2.0]
print(f"Prompt: {prompt}")
for penalty in penalties:
    print()
    print(f"Repetition penalty: {penalty}")
    output = model.generate(
        **inputs,
        max_length=100,
        num_return_sequences=1,
        temperature=0.3,
        top_k=50,
        top_p=1.0,
        repetition_penalty=penalty,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print("Generated Text:")
    print(generated_text)

Prompt: Once upon a time, there was a

Repetition penalty: 1.0
Generated Text:
Once upon a time, there was a man in the middle of the street who had a gun. He was armed with a .45 caliber pistol. He was carrying a .45 caliber pistol. He said he was going to kill the police officer. He said he was going to kill the police officer. He said he was going to kill the police officer. He said he was going to kill the police officer. He said he was going to kill the police officer. He said he was going to

Repetition penalty: 1.2
Generated Text:
Once upon a time, there was a man named Ockham who had been in the army for about twenty years and could not be found. He told me that he came to England from France with his wife when they were young men of twelve or thirteen; but after having served some one year as an officer at London's Royal Navy Yard before being sent back again by Captain Thomas Smith (who then went on to become Chief Warrant Officer) it became clear how much more difficult this

## Zachłanne dekodowanie i próbkowanie


Parametr do_sample określa, czy model używa próbkowania (probabilistycznego wyboru tokenów), czy zachłannego dekodowania (zawsze wybierania najbardziej prawdopodobnego tokenu). Porównajmy te podejścia:

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "The secret to happiness is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text with greedy decoding vs. sampling
print(f"Prompt: {prompt}\n")
print("Greedy Decoding (do_sample=False):")
output = model.generate(
    **inputs,
    max_length=100,
    num_return_sequences=1,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("Generated Text:")
print(generated_text)
print()
print("Sampling (do_sample=True):")
output = model.generate(
    **inputs,
    max_length=100,
    num_return_sequences=1,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("Generated Text:")
print(generated_text)

Prompt: The secret to happiness is

Greedy Decoding (do_sample=False):
Generated Text:
The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret to happiness is to be happy.

The secret to happiness is to be happy.

The

Sampling (do_sample=True):
Generated Text:
The secret to happiness is that people aren't happy to the point that they won't even try when faced with the prospect of death, which is also why there are no death panels and no death-related messages, despite the fact that we live in the moment. It's all because of the things that people find interesting in life.

I've always been fascinated with why people can be happy because they're the lucky ones, because I found things that have made me change my entire life;


Spróbuj uruchomić ten kod wiele razy i obserwować dane wyjściowe. Zauważysz, że wynik zachłannego dekodowania jest zawsze taki sam, podczas gdy wynik próbkowania jest za każdym razem inny. Dekodowanie zachłanne jest deterministyczne dla ustalonego monitu. Model generuje rozkład prawdopodobieństwa i wybierany jest najbardziej prawdopodobny token. Nie ma tu żadnej losowości. Jest bardziej prawdopodobne, że dane wyjściowe będą powtarzalne i nieprzydatne.

Dane wyjściowe próbkowania są stochastyczne, ponieważ tokeny wyjściowe są wybierane na podstawie przewidywanego rozkładu prawdopodobieństwa modelu. Losowość pozwala modelowi generować bardziej zróżnicowany i kreatywny tekst, podczas gdy dane wyjściowe są nadal spójne, o ile inne parametry generowania są ustawione prawidłowo. W przypadku próbkowania wyjściowego można ustawić num_return_sequences na liczbę większą niż 1, aby wygenerować wiele sekwencji równolegle dla tego samego monitu. Ten parametr nie ma znaczenia dla zachłannego dekodowania.


## Parametry dla konkretnych zastosowań


Jakie wartości parametrów należy ustawić dla konkretnej aplikacji? Nie ma konkretnej odpowiedzi. Z pewnością musisz przeprowadzić kilka eksperymentów, aby znaleźć najlepsze kombinacje. Ale możesz użyć następujących elementów jako punktu wyjścia:


Generowanie faktów:
* Niższa temperatura (od 0,2 do 0,4) dla bardziej deterministycznego wyjścia
* Umiarkowane top_p (od 0,8 do 0,9), aby odfiltrować mało prawdopodobne tokeny
* Wyższe repetition_penalty (od 1,2 do 1,5), aby uniknąć powtarzających się stwierdzeń

Kreatywne pisanie:
* Wyższa temperatura (od 1,0 do 1,3) zapewnia bardziej kreatywną i zróżnicowaną wydajność
* Wyższe top_p (od 0,9 do 0,95), aby zapewnić więcej możliwości
* Niższe repetition_penalty (od 1,0 do 1,1), aby umożliwić pewne powtórzenia stylistyczne

Generowanie kodu:
* Niższa temperatura (od 0,1 do 0,3) dla bardziej precyzyjnego i poprawnego kodowania
* Niższe top_p (od 0,7 do 0,8), aby skupić się na najbardziej prawdopodobnych tokenach
* Wyższe repetition_penalty (od 1,3 do 1,5), aby uniknąć nadmiarowego kodu

Generowanie dialogów:
* Umiarkowana temperatura (od 0,6 do 0,8) dla naturalnych, ale skoncentrowanych reakcji
* Umiarkowane top_p (0,9) dla dobrej równowagi między kreatywnością a spójnością
* Moderuj repetition_penalty (1.2), aby uniknąć powtarzających się fraz.

Pamiętaj, że model językowy nie jest idealną wyrocznią. Może popełniać błędy. Powyższe parametry mają na celu pomóc w dopasowaniu procesu generowania do oczekiwanego stylu wyjściowego, ale nie gwarantować poprawności. Otrzymane dane wyjściowe mogą zawierać błędy.

## Wyszukiwanie wiązki/Beam Search i generowanie wielu sekwencji

W powyższych przykładach proces generowania jest autoregresyjny. Jest to proces iteracyjny, który generuje jeden token na raz.

Ponieważ każdy krok generuje jeden token poprzez próbkowanie, nic nie stoi na przeszkodzie, aby wygenerować wiele tokenów jednocześnie. Jeśli to zrobisz, wygenerujesz wiele sekwencji wyjściowych dla jednego monitu wejściowego. Teoretycznie, jeśli wygenerujesz $k$
tokenów na każdym kroku i ustawiasz długość n, to wygenerujesz $k^n$
sekwencji. Może to być duża liczba i możesz chcieć ograniczyć ją tylko do kilku.

Pierwszym sposobem generowania wielu sekwencji jest ustawienie **num_return_sequences** na liczbę k. Generujesz k
tokenów w pierwszym kroku. Następnie wykonaj sekwencję dla każdego z nich. To zasadniczo powiela monit k razy.

Drugim sposobem jest użycie wyszukiwania wiązki. Jest to bardziej wyrafinowany sposób generowania wielu sekwencji. Śledzi najbardziej obiecujące sekwencje i bada je równolegle. Zamiast generować $k^n$
sekwencji, aby przytłoczyć pamięć, zachowuje tylko $k$
najlepszych sekwencji na każdym kroku. Każdy krok generowania tokenów tymczasowo rozszerzy ten zestaw i przytnie go z powrotem do k
najlepszych sekwencji.

Aby użyć wyszukiwania wiązek, należy ustawić num_beams na liczbę $k$. Każdy krok rozszerzy każdą z $k$
 sekwencji o jeszcze jeden token, uzyskując $k^2$
 sekwencji, a następnie wybierze najlepsze $k$
 sekwencji, aby przejść do następnego kroku. Można również ustawić early_stopping=True, aby zatrzymać generowanie po osiągnięciu końca sekwencji. Należy również ustawić num_return_sequences, aby ograniczyć ostateczny wybór na wyjściu.


Wybór sekwencji jest zwykle oparty na skumulowanym prawdopodobieństwie tokenów w sekwencji. Ale możesz również zniekształcić wybór według innych kryteriów, takich jak dodanie kary za długość lub unikanie powtarzania n-gramów. Poniżej znajduje się przykład użycia wyszukiwania wiązki:

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "The key to successful machine learning is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text with greedy decoding vs. sampling
print(f"Prompt: {prompt}\n")
outputs = model.generate(
    **inputs,
    num_beams=5,             # Number of beams to use
    early_stopping=True,     # Stop when all beams have finished
    no_repeat_ngram_size=2,  # Avoid repeating n-grams
    num_return_sequences=3,  # Return multiple sequences
    max_length=100,
    temperature=1.5,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
for idx, output in enumerate(outputs):
    generated_text = tokenizer.decode(output, skip_special_tokens=True)
    print(f"Generated Text ({idx+1}):")
    print(generated_text)

Prompt: The key to successful machine learning is

Generated Text (1):
The key to successful machine learning is the ability to predict the future. In order to do this, we need to start with a set of assumptions.

We are not going to pretend that we know everything about your computer. But we do know a lot of things about it. We know that it does not have all the bells and whistles that you would expect. For example, if you were to ask yourself, "What is your favorite computer?" you wouldn't be able to tell what
Generated Text (2):
The key to successful machine learning is the ability to predict the future. In order to do this, we need to start with a set of assumptions.

We are not going to pretend that we know everything about your computer. But we do know a lot of things about it. We know that it does not have all the bells and whistles that you would expect. For example, if you were to ask yourself, "What is your favorite computer?" you wouldn't be able to tell because
Generated Te

Liczba sekwencji wyjściowych jest nadal kontrolowana przez num_return_sequences, ale proces ich generowania wykorzystuje algorytm wyszukiwania wiązki. Nie jest łatwo określić, czy na podstawie danych wyjściowych używane jest wyszukiwanie wiązki. Jednym ze znaków jest to, że wyniki wyszukiwania wiązki nie są tak różnorodne, jak samo ustawienie num_return_sequences, ponieważ generowanych jest znacznie więcej sekwencji i wybierane są te o wyższym skumulowanym prawdopodobieństwie. To filtrowanie rzeczywiście zmniejszyło różnorodność danych wyjściowych.